In [ ]:
# =========================================================
# FUNCIONES TEMPORALES DE FORWARD KINEMATICS
# PARA QUE EL CODIGO COMPLETE FUNCIONE
# =========================================================

import numpy as np
from math import cos, sin, radians

# ---------------------------------------------------------
# FORWARD KINEMATICS
# ---------------------------------------------------------

def forward_kinematics(angles):
    """
    Calcula una matriz homogénea simplificada del MyCobot.

    Parametros
    ----------
    angles : list[float]
        Lista de 6 angulos en grados

    Retorna
    -------
    T : np.ndarray (4x4)
        Matriz de transformacion homogénea
    """

    # ---------------------------------------------
    # Convertir grados a radianes
    # ---------------------------------------------

    theta1 = radians(angles[0])
    theta2 = radians(angles[1])
    theta3 = radians(angles[2])

    # ---------------------------------------------
    # Longitudes aproximadas de eslabones (mm)
    # ---------------------------------------------

    L1 = 131.0
    L2 = 110.0
    L3 = 96.0

    # ---------------------------------------------
    # Posicion simplificada del efector final
    # ---------------------------------------------

    r = (
        L2 * cos(theta2) +
        L3 * cos(theta2 + theta3)
    )

    x = r * cos(theta1)

    y = r * sin(theta1)

    z = (
        L1 +
        L2 * sin(theta2) +
        L3 * sin(theta2 + theta3)
    )

    # ---------------------------------------------
    # Crear matriz homogénea 4x4
    # ---------------------------------------------

    T = np.eye(4)

    T[0, 3] = x
    T[1, 3] = y
    T[2, 3] = z

    return T


# ---------------------------------------------------------
# EXTRAER POSICION XYZ
# ---------------------------------------------------------

def extract_position(T):
    """
    Extrae coordenadas XYZ desde matriz homogénea.

    Parametros
    ----------
    T : np.ndarray
        Matriz 4x4

    Retorna
    -------
    (x, y, z)
    """

    x = T[0, 3]
    y = T[1, 3]
    z = T[2, 3]

    return x, y, z

In [ ]:
#!/usr/bin/env python3
# coding: utf-8

"""
P4 - Evasion de Colisiones (Problema Abierto)
MyCobot 280 - 6 DOF
"""

import sys
import os
import time
import logging
import numpy as np

from math import (
    sqrt,
    cos,
    sin,
    radians
)

logger = logging.getLogger(__name__)

# =========================================================
# FORWARD KINEMATICS
# =========================================================

def forward_kinematics(angles):

    theta1 = radians(angles[0])
    theta2 = radians(angles[1])
    theta3 = radians(angles[2])

    # Longitudes aproximadas del MyCobot
    L1 = 131.0
    L2 = 110.0
    L3 = 96.0

    # Calculo simplificado
    r = (
        L2 * cos(theta2) +
        L3 * cos(theta2 + theta3)
    )

    x = r * cos(theta1)

    y = r * sin(theta1)

    z = (
        L1 +
        L2 * sin(theta2) +
        L3 * sin(theta2 + theta3)
    )

    # Matriz homogénea
    T = np.eye(4)

    T[0, 3] = x
    T[1, 3] = y
    T[2, 3] = z

    return T


# =========================================================
# EXTRAER POSICION XYZ
# =========================================================

def extract_position(T):

    x = T[0, 3]
    y = T[1, 3]
    z = T[2, 3]

    return x, y, z

In [ ]:
"""
Estrategia elegida:

1. Limites conservadores por joint
2. Verificacion de altura minima usando FK
3. Waypoint seguro de clearance

Justificacion:
- Evita choques con la mesa
- Reduce auto-colisiones
- Evita colisiones con objetos del laboratorio
"""

In [ ]:
COLLISION_SCENARIOS = {

    "colision_mesa": {
        "descripcion": "El extremo impacta la mesa.",
        "causa": "Angulos J2/J3 generan Z muy baja.",
        "prevencion": "Verificar altura minima via FK."
    },

    "auto_colision_J2_J3": {
        "descripcion": "Colision entre eslabones.",
        "causa": "Configuraciones extremas de J2 y J3.",
        "prevencion": "Limites conservadores."
    },

    "colision_camara": {
        "descripcion": "El gripper impacta la camara.",
        "causa": "Trayectorias directas.",
        "prevencion": "Waypoint intermedio seguro."
    },

    "colision_zona_deposito": {
        "descripcion": "Choque con contenedores.",
        "causa": "Movimiento lateral a baja altura.",
        "prevencion": "Elevar Z antes de mover."
    }
}

In [ ]:
CONSERVATIVE_JOINT_LIMITS = {

    0: (-160, 160),
    1: (-110, 70),
    2: (-120, 120),
    3: (-130, 130),
    4: (-150, 150),
    5: (-175, 175),
}

# Altura minima segura
Z_MIN_SAFE = 60.0

# Altura de clearance
Z_CLEARANCE = 200.0

# Waypoint seguro
SAFE_WAYPOINT_ANGLES = [
    0.0,
    0.0,
    -90.0,
    95.0,
    0.0,
    -45.0
]

In [ ]:
class CollisionChecker:

    def __init__(self, mc=None):

        self.mc = mc
        self.joint_limits = CONSERVATIVE_JOINT_LIMITS
        self.z_min = Z_MIN_SAFE
        self.z_clearance = Z_CLEARANCE

In [ ]:
def check_joint_limits(self, angles):

        for i, angle in enumerate(angles):

            lo, hi = self.joint_limits[i]

            if not (lo <= angle <= hi):

                msg = (
                    f"J{i+1}={angle:.1f} "
                    f"fuera de limite [{lo},{hi}]"
                )

                return False, msg

        return True, "Todos los joints OK"

In [ ]:
def check_min_height(self, angles):

        T = forward_kinematics(angles)

        x, y, z = extract_position(T)

        if z < self.z_min:

            msg = (
                f"Z={z:.1f} mm "
                f"< Zmin={self.z_min:.1f} mm"
            )

            return False, z, msg

        return True, z, f"Z={z:.1f} mm OK"

In [ ]:
def safe_move(self, target_angles, speed=40):

        # ---------------------------------
        # 1. Verificar limites
        # ---------------------------------

        valid_limits, msg_limits = \
            self.check_joint_limits(target_angles)

        if not valid_limits:

            logger.warning(msg_limits)

            return False

        # ---------------------------------
        # 2. Verificar altura minima
        # ---------------------------------

        valid_z, z_val, msg_z = \
            self.check_min_height(target_angles)

        if not valid_z:

            logger.warning(msg_z)

            return False

        # ---------------------------------
        # 3. Ejecutar movimiento seguro
        # ---------------------------------

        if self.mc is not None:

            logger.info(
                "Subiendo a waypoint seguro..."
            )

            self.mc.send_angles(
                SAFE_WAYPOINT_ANGLES,
                speed
            )

            time.sleep(2.5)

            logger.info(
                f"Moviendo a destino Z={z_val:.1f}"
            )

            self.mc.send_angles(
                target_angles,
                speed
            )

            time.sleep(2.5)

        else:

            logger.info(
                f"[OFFLINE] Movimiento valido "
                f"Z={z_val:.1f}"
            )

        return True

In [ ]:
def safe_coords(self, target_coords, speed=40):

        x, y, z = (
            target_coords[0],
            target_coords[1],
            target_coords[2]
        )

        # Verificar altura
        if z < self.z_min:

            logger.warning(
                f"Coords BLOQUEADAS "
                f"Z={z:.1f}"
            )

            return False

        if self.mc is not None:

            current = self.mc.get_coords()

            # Elevar primero
            if current and len(current) >= 6:

                elevated = list(current)

                elevated[2] = self.z_clearance

                logger.info(
                    f"Elevando a "
                    f"Z={self.z_clearance}"
                )

                self.mc.send_coords(
                    elevated,
                    speed,
                    1
                )

                time.sleep(2.0)

            logger.info(
                f"Moviendo a "
                f"[{x:.1f},{y:.1f},{z:.1f}]"
            )

            self.mc.send_coords(
                target_coords,
                speed,
                1
            )

            time.sleep(2.5)

        else:

            logger.info(
                f"[OFFLINE] Coordenadas validas "
                f"Z={z:.1f}"
            )

        return True

In [ ]:
def validate_with_real_robot(mc):

    checker = CollisionChecker(mc=mc)

    test_sequences = [

        ("Pose home",
         [0,0,0,0,0,-45]),

        ("Watch pose",
         [42,0,0,-85,-7,-3]),

        ("Limite J2",
         [0,-110,0,0,0,0]),

        ("Fuera de limite",
         [0,-120,0,0,0,0]),

        ("Z muy baja",
         [0,90,90,90,0,0]),
    ]

    print("="*60)
    print("VALIDACION ROBOT REAL")
    print("="*60)

    for desc, angles in test_sequences:

        print(f"\n[{desc}]")

        valid_l, msg_l = \
            checker.check_joint_limits(angles)

        valid_z, z_val, msg_z = \
            checker.check_min_height(angles)

        print(msg_l)
        print(msg_z)

        if valid_l and valid_z:

            result = checker.safe_move(angles)

            print(f"Movimiento: {result}")

        else:

            print("Movimiento bloqueado")

In [ ]:
checker = CollisionChecker(mc=None)

test_cases = [

    (
        [0,0,0,0,0,-45],
        "Pose home"
    ),

    (
        [0,-120,0,0,0,0],
        "Fuera de limites"
    ),

    (
        [0,80,80,90,0,0],
        "Z demasiado baja"
    )
]

print("\nPruebas offline:")

for angles, desc in test_cases:

    valid_l, msg_l = \
        checker.check_joint_limits(angles)

    valid_z, z_val, msg_z = \
        checker.check_min_height(angles)

    status = \
        "PASO" if (valid_l and valid_z) \
        else "BLOQUEADO"

    print(f"\n[{status}] {desc}")

    if not valid_l:
        print(msg_l)

    if not valid_z:
        print(msg_z)

In [ ]:
if __name__ == "__main__":

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )

    print("="*70)
    print("P4 - EVASION DE COLISIONES")
    print("="*70)

    print("\nEscenarios de colision:")

    for name, info in COLLISION_SCENARIOS.items():

        print(f"\n[{name}]")

        print(
            f"Descripcion: "
            f"{info['descripcion']}"
        )

        print(
            f"Causa: "
            f"{info['causa']}"
        )

        print(
            f"Prevencion: "
            f"{info['prevencion']}"
        )

    print("\nLimites conservadores:")

    for i, (lo, hi) in \
            CONSERVATIVE_JOINT_LIMITS.items():

        print(
            f"J{i+1}: "
            f"[{lo}, {hi}]"
        )

    # Ejecutar pruebas offline
    print("\nEjecutando pruebas...")